In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *
from pyspark.sql.types import StringType, MapType

In [0]:
@dp.temporary_view(
    name="customers_bronze_view",
    comment="Standardized view of bronze customers with unified column names"
)
def customers_bronze_view():
    """
    Temporary view that standardizes customer data from bronze layer.
    Handles multiple column name variants using COALESCE to merge into standard schema.
    """
    df =  (
        spark.readStream.table("customers")
        .select(
            # Standardize customer_id from variants: customer_id, CustomerID, cust_id, customer_identifier
            coalesce(
                col("customer_id"),
                col("CustomerID"),
                col("cust_id"),
                col("customer_identifier")
            ).alias("customer_id"),
            
            # Standardize customer_email from variants: customer_email, email_address
            coalesce(
                col("customer_email"),
                col("email_address")
            ).alias("customer_email"),
            
            # Standardize customer_name from variants: customer_name, full_name
            coalesce(
                col("customer_name"),
                col("full_name")
            ).alias("customer_name"),
            
            # Standardize segment from variants: segment, customer_segment
            coalesce(
                col("segment"),
                col("customer_segment")
            ).alias("segment"),
            
            # These columns are already standardized
            col("country"),
            col("city"),
            col("state"),
            col("postal_code"),
            col("region"),
            col("source_region"),
            col("_rescued_data"), 
            col("_source_file_path"), 
            col("_source_file_name"), 
            col("_source_modified_time"), 
            col("_ingested_at")
        )
    )
    return df


In [0]:
@dp.temporary_view(
    name="returns_bronze_view",
    comment="Standardized view of bronze returns with unified column names"
)
def returns_bronze_view():
    """
    Temporary view that standardizes returns data from bronze layer.
    Handles multiple column name variants using COALESCE to merge into standard schema.
    """
    return (
        spark.readStream.table("returns")
        .select(
            # Standardize order_id from variants: order_id, OrderId
            coalesce(
                col("order_id"),
                col("OrderId")
            ).alias("order_id"),
            
            # Standardize return_reason from variants: return_reason, reason
            coalesce(
                col("return_reason"),
                col("reason")
            ).alias("return_reason"),
            
            # Standardize return_date from variants: return_date, date_of_return
            coalesce(
                col("return_date"),
                col("date_of_return")
            ).alias("return_date"),
            
            # Standardize refund_amount from variants: refund_amount, amount
            coalesce(
                col("refund_amount"),
                col("amount")
            ).alias("refund_amount"),
            
            # Standardize return_status from variants: return_status, status
            coalesce(
                col("return_status"),
                col("status")
            ).alias("return_status"),
            col("_rescued_data"), 
            col("_source_file_path"), 
            col("_source_file_name"), 
            col("_source_modified_time"), 
            col("_ingested_at")
        )
    )

In [0]:
@dp.temporary_view(
    name="transactions_bronze_view",
    comment="Standardized view of bronze transactions with retrieved column names from _rescued_data column"
)
def transactions_bronze_view():
    df = spark.readStream.table("transactions")
    df = df.withColumn("rescued_map", from_json(col("_rescued_data"), MapType(StringType(), StringType())))
    df = df.withColumns({
                            "order_id": when(
                                col("_rescued_data").isNotNull(),
                                coalesce(
                                    col("order_id"),
                                    col("rescued_map")["Order_ID"]
                                )
                            ).otherwise(col("order_id")),

                            "product_id": when(
                                col("_rescued_data").isNotNull(),
                                coalesce(
                                    col("product_id"),
                                    col("rescued_map")["Product_ID"]
                                )
                            ).otherwise(col("product_id"))
                            })

    df = df.drop("rescued_map")

    return df